# v2 — From a Lookup Table to a Linear Head (+ Position Embeddings)

This builds directly on **v1**. Steps 1–7 below (imports, hyperparameters, data, tokenizer, train/val split, batching, loss estimation) are *unchanged* from v1 — copy them straight over.

The real change is in the model (step 8):

1. `token_embedding_table` now maps a token to an `emb_size`-dim vector, instead of directly to `vocab_size` logits.
2. We add a **position embedding** table — the model can now (in principle) tell *where* in the block a token sits. It won't do much yet, since a bigram model has no mechanism to use that extra context — but every block we add from here on (attention, etc.) builds on this same pattern.
3. A `linear_head` projects the embedding back up to `vocab_size` logits.
4. Because position embeddings only go up to `block_size`, `generate` must now crop the context to the last `block_size` tokens before calling the model.

## 1. Imports (same as v1)

In [ ]:
import torch
import torch.nn as nn
from torch.nn import functional as F

## 2. Hyperparameters

Same as v1, plus one new one: `emb_size` — the dimensionality of the token/position embedding vectors. This is what decouples the embedding size from `vocab_size`.

In [ ]:
emb_size = 32        # NEW: embedding size for each token
batch_size = 32      # how many independent sequences will we process in parallel?
block_size = 8       # what is the maximum context length for predictions?
max_iters = 10000    # maximum number of iterations for training
eval_interval = 500  # interval for evaluating the model
learning_rate = 1e-3 # learning rate for training
device = 'cuda' if torch.cuda.is_available() else 'cpu'
eval_iters = 200     # number of iterations for evaluation
seed = 42            # seed for reproducibility

torch.manual_seed(seed)

## 3. Load the dataset

In [ ]:
with open('./data/harry_potter.txt', encoding='utf-8') as f:
    text = f.read()

print(f"length of dataset in characters: {len(text)}")
print(text[:500])

## 4. Tokenization: characters as tokens

In [ ]:
chars = sorted(list(set(text)))
vocab_size = len(chars)
print(''.join(chars))
print(vocab_size)

In [ ]:
stoi = { ch:i for i,ch in enumerate(chars) }
itos = { i:ch for i,ch in enumerate(chars) }
encode = lambda s: [stoi[c] for c in s] # encoder: take a string, output a list of integers
decode = lambda l: ''.join([itos[i] for i in l]) # decoder: take a list of integers, output a string

print(encode("Hello there!"))
print(decode(encode("Hello there!")))

## 5. Train / validation split

In [ ]:
data = torch.tensor(encode(text), dtype=torch.long)
n = int(0.9*len(data)) # first 90% will be train, rest val
train_data = data[:n]
val_data = data[n:]

print(data.shape, data.dtype)
print(data[:100])

## 6. Batching

In [ ]:
def get_batch(split):
    # generate a small batch of data of inputs x and targets y
    data = train_data if split == 'train' else val_data
    ix = torch.randint(len(data) - block_size, (batch_size,))
    x = torch.stack([data[i:i+block_size] for i in ix])
    y = torch.stack([data[i+1:i+block_size+1] for i in ix])
    x, y = x.to(device), y.to(device)
    return x, y

In [ ]:
xb, yb = get_batch('train')
print('inputs:')
print(xb.shape)
print(xb)
print('targets:')
print(yb.shape)
print(yb)

## 7. Estimating loss

In [ ]:
@torch.no_grad()
def estimate_loss():
    out = {}
    model.eval()
    for split in ['train', 'val']:
        losses = torch.zeros(eval_iters)
        for k in range(eval_iters):
            X, Y = get_batch(split)
            logits, loss = model(X, Y)
            losses[k] = loss.item()
        out[split] = losses.mean()
    model.train()
    return out

## 8. The model — what's new

In v1, `token_embedding_table` mapped a token directly to its `vocab_size` logits. Now we decouple the embedding dimension from the vocabulary size:

- `token_embedding_table`: token id → `emb_size`-dim vector *(changed: used to go straight to `vocab_size`)*
- `position_embedding_table` **(new)**: position id → `emb_size`-dim vector, so the model has a notion of *where* a token sits in the block
- `linear_head` **(new)**: projects the summed token + position embedding back up to `vocab_size` logits
- in `generate`, the context must now be cropped to `block_size` **(new)**, since `position_embedding_table` only has `block_size` rows

**In class:** fill in the parts marked **NEW** below — everything else carries over unchanged from v1.

In [ ]:
# bigram model with a linear head and position embeddings
class BigramLanguageModel(nn.Module):

    def __init__(self, vocab_size, emb_size):
        super().__init__()
        # NEW: each token now maps to an emb_size-dim vector, not directly to vocab_size logits
        self.token_embedding_table = ...

        # NEW: position id (0..block_size-1) -> emb_size-dim vector
        # we'll make real use of this once we add attention
        self.position_embedding_table = ...

        # NEW: projects the emb_size-dim vector back up to vocab_size logits
        self.linear_head = ...

    def forward(self, idx, targets=None):
        B, T = idx.shape

        # idx and targets are both (B,T) tensor of integers
        # NEW: look up token embeddings -> (B,T,emb_size)
        tok_emb = ...
        # C is the embedding size, T is the sequence length, B is the batch size
        # NEW: look up position embeddings for positions 0..T-1 -> (T,emb_size)
        pos_emb = ...
        # NEW: combine token identity and position -> (B,T,emb_size)
        x = ...
        # NEW: project up to vocab_size logits -> (B,T,vocab_size)
        logits = ...

        if targets is None:
            loss = None
        else:
            B, T, C = logits.shape
            logits = logits.view(B*T, C)
            targets = targets.view(B*T)
            loss = F.cross_entropy(logits, targets)

        return logits, loss

    def generate(self, idx, max_new_tokens):
        # idx is (B, T) array of indices in the current context
        for _ in range(max_new_tokens):
            # NEW: crop idx to the last block_size tokens
            # (position_embedding_table only has block_size rows!)
            idx_cond = ...

            # get the predictions
            logits, loss = self(idx_cond)
            # focus only on the last time step
            logits = logits[:, -1, :] # becomes (B, C)
            # apply softmax to get probabilities
            probs = F.softmax(logits, dim=-1) # (B, C)
            # sample from the distribution
            idx_next = torch.multinomial(probs, num_samples=1) # (B, 1)
            # append sampled index to the running sequence
            idx = torch.cat((idx, idx_next), dim=1) # (B, T+1)
        return idx

## 9. Sanity check: untrained generation

Note the model now also takes `emb_size` when constructed.

In [ ]:
model = BigramLanguageModel(vocab_size, emb_size)
model = model.to(device)
# print the number of parameters in the model
print(sum(p.numel() for p in model.parameters())/1e6, 'M parameters')

context = torch.zeros((1, 1), dtype=torch.long, device=device)
print(decode(model.generate(context, max_new_tokens=200)[0].tolist()))

## 10. Training the model

In [ ]:
# create a PyTorch optimizer
optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)

for iter in range(max_iters):

    # every once in a while evaluate the loss on train and val sets
    if iter % eval_interval == 0 or iter == max_iters - 1:
        losses = estimate_loss()
        print(f"step {iter}: train loss {losses['train']:.4f}, val loss {losses['val']:.4f}")

    # sample a batch of data
    xb, yb = get_batch('train')

    # evaluate the loss
    logits, loss = model(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()

## 11. Generate text from the trained model

In [ ]:
print('''\n##########################################
# Let's generate some Harry Potter text! #
##########################################''')
context = torch.zeros((1, 1), dtype=torch.long, device=device)
print(decode(model.generate(context, max_new_tokens=1000)[0].tolist()))